### Factor Decomposition for an S&P Portfolio

In [4]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src import data, factors, plotting

spy_px, spy_ret = data.get_market()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
plotting.plot_stock(prices, "AAPL")
plotting.plot_vs_market(prices, spy_px, "AAPL")

### Single-Factor Model: A Naive Look at Idiosyncratic Returns

We decompose a stock's daily return into a market component and a stock-specific component:

$$
r_{i,t} = \alpha_i + \beta_i \, r_{m,t} + \varepsilon_{i,t}
$$

- $r_{i,t}$: return of stock $i$ on day $t$
- $r_{m,t}$: market (SPY) return on day $t$
- $\beta_i = \dfrac{\mathrm{Cov}(r_i, r_m)}{\mathrm{Var}(r_m)}$: sensitivity to the market
- $\alpha_i$: average return not explained by the market
- $\varepsilon_{i,t}$: residual, the **idiosyncratic return**

The idiosyncratic return is what's left after stripping out the market:

$$
\varepsilon_{i,t} = r_{i,t} - \alpha_i - \beta_i \, r_{m,t}
$$

This also splits total variance into systematic and idiosyncratic parts (the residual is uncorrelated with the market by construction):

$$
\mathrm{Var}(r_i) = \beta_i^2 \, \mathrm{Var}(r_m) + \mathrm{Var}(\varepsilon_i)
$$

**Why it's naive:** the market is the only factor. Anything driven by sector or style (size, momentum, volatility) gets lumped into $\varepsilon$, so this overstates true idiosyncratic risk. A multi-factor model peels those layers off next.

In [12]:
plotting.plot_regression(prices, spy_px, "AAPL")

In [13]:
qqq_px, qqq_ret = data.get_market("QQQ")
plotting.plot_regression(prices, qqq_px, "AAPL", market_name="QQQ")

In [14]:
iwm_px, _ = data.get_market("IWM")   # Russell 2000 (small caps)
plotting.plot_regression(prices, iwm_px, "AAPL", market_name="IWM")

### Two-Factor Model: Market + Sector

Extend the single-factor model with a sector return (e.g. XLK for Apple):

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, r_{s,t} + \varepsilon_{i,t}
$$

**Problem: collinearity.** Sector and market returns are highly correlated ($\rho \approx 0.85$ to $0.9$ for XLK vs SPY), so the two betas are unstable and hard to interpret.

**Fix: orthogonalize the sector against the market.** First strip the market out of the sector:

$$
r_{s,t} = a + b \, r_{m,t} + u_t
$$

Then regress the stock on the market and the sector residual $u_t$:

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, u_t + \varepsilon_{i,t}
$$

- $u_t$: the sector-specific move, i.e. what the sector did beyond the market
- $\beta_m$: identical to the single-factor beta, since $u$ is uncorrelated with $r_m$
- $\beta_s$: sensitivity to the sector-specific move
- $\Delta R^2$ vs the single-factor model: variance explained by the sector on top of the market

**Caveat: self-inclusion.** Apple is a large weight in XLK, so part of the "sector" is Apple itself, which inflates $\beta_s$ and $R^2$. Cleaner proxies are an equal-weight sector ETF or the average return of the other stocks in the sector, excluding the one being modeled.

___

### Orthogonalization: The Idea

Treat each return series as a vector, one entry per day. Two series that move together point in similar directions. That overlap is what makes their effects hard to separate in a regression.

Orthogonalizing $s$ against $m$ splits $s$ into two pieces:

$$
s = \underbrace{b \, m}_{\text{part explained by } m} + \underbrace{u}_{\text{part unrelated to } m}
$$

- $b \, m$ is the **projection** of $s$ onto $m$: the "shadow" $s$ casts in the market's direction, with $b = \dfrac{\mathrm{Cov}(s, m)}{\mathrm{Var}(m)}$
- $u = s - b \, m$ is what's left, and by construction $\mathrm{Cov}(u, m) = 0$

Geometrically, $u$ is perpendicular to $m$: it carries no information about the market at all.

**Why it helps:** the two regressors $m$ and $u$ no longer overlap, so each coefficient is cleanly identified. The market beta is unchanged from the single-factor model, and the sector coefficient captures only what's new, i.e. the sector's own move with the market stripped out.

**The catch:** the result depends on order. Orthogonalizing the sector against the market gives the market "first claim" on the shared variance. Reversing the order would credit that shared variance to the sector instead.
___